# Exercise 5
## Model fitting
*** 

### Question
How can I fit a model to my dataset?

### Objectives
1. Investigate simple line fitting.
2. Plot data with errors.
3. Learn to use minimisation algorithms to fit more general (non-linear) models to data.

**Revise:**
- Reading in and plotting data;
- Defining functions;
- Map plotting.

### Independent coding
Use a minimisation algorithm to find the optimal pole of rotation between South America and Africa.

**Run this cell first.** It loads `check()`, which we use to check your answers to the **Your turn** tasks: fill in the `___` blanks, run the cell, and `check()` tells you whether you're right.

In [ ]:
from check import check, ___

## 5.1 Fitting a line to data
***

### 5.1.1 Scientific background

Fitting a model to some data is a very common task in the sciences. We have a model (in practice a function) we believe fits some data, subject to us specifying the right set of parameters.  The task is to identify the model parameters that enable the model to best fit the data - often these parameters will be what we are trying to identify  in the first place, so the 'fitting' procedure is actually at the heart of the science.  

An example you will probably be familiar with is isochrons.  As a reminder, an isochron equation is written below for the Rb-Sr system, where $^\mathrm{87}\mathrm{Rb}$ decays to $^\mathrm{87}\mathrm{Sr}$ with a half life of about 49.8 billion years,

\begin{align}
\frac{^\mathrm{87}\mathrm{Sr}}{^\mathrm{86}\mathrm{Sr}} &= \frac{^\mathrm{87}\mathrm{Sr}}{^\mathrm{86}\mathrm{Sr}}\bigg\rvert_0 + 
\frac{^\mathrm{87}\mathrm{Rb}}{^\mathrm{86}\mathrm{Sr}}\big(\exp{(\lambda{}t)-1}\big),\\\\
\lambda &= -\frac{\ln{1/2}}{\tau_{1/2}}.
\end{align}

Where in these equations $\frac{^\mathrm{87}\mathrm{Sr}}{^\mathrm{86}\mathrm{Sr}}$ is the present Sr isotopic composition of the material, $\frac{^\mathrm{87}\mathrm{Sr}}{^\mathrm{86}\mathrm{Sr}}\big\rvert_0$ was the initial Sr isotope composition in the material when the chronometer was last reset, $\frac{^\mathrm{87}\mathrm{Rb}}{^\mathrm{86}\mathrm{Sr}}$ is the present ratio of the parent isotope, $^\mathrm{87}\mathrm{Rb}$, to the stable reference isotope $^\mathrm{86}\mathrm{Sr}$, and $\lambda$ is the decay constant, which is related to the half life $\tau_{1/2}$.  This equation is simply telling us how the Sr isotope composition of a material evolves with time.

The number we are most interested in extracting from the isochron equation is $t$, the time since the isotope system was last reset.  For example, for a volcanic ash layer (ash layers are commonly used to provide absolute ages for sediment cores), this time will be the age at which the minerals in the ash layer cooled through their closure temperature, i.e., the eruption age.  

The utility of the isochron equation is that we can measure both the Sr isotopic composition and Rb abundance of a sample at the present, and when we plot those data on a graph we should obtain a familiar linear relationship, $y=mx+c$, where in this case

\begin{align}
y &= \frac{^\mathrm{87}\mathrm{Sr}}{^\mathrm{86}\mathrm{Sr}},\ &x &= \frac{^\mathrm{87}\mathrm{Rb}}{^\mathrm{86}\mathrm{Sr}}\\
m &= \exp{(\lambda{}t)-1},\ &c &= \frac{^\mathrm{87}\mathrm{Sr}}{^\mathrm{86}\mathrm{Sr}}\bigg\rvert_0.
\end{align}

So as an exercise in model fitting, all we need to do is find the 'best' line through the data and we can obtain both the initial Sr isotopic composition of the material and its age. 

### 5.1.2 Describing how good a line is at fitting isochron data

From a computational perspective this is a special case of a minimisation/optimisation problem.  It is a minimisation problem in that we want to minimise the misfit between a model and the data, i.e., in the case of an isochron, have the straight line go as close as possible to all the points.

Let's illustrate how we might do this in Python with an Rb-Sr isotopic dataset.  We are going to be looking at data from the Sierra Nevada Granodiorite, published in __[Kistler et al. (1986)](https://link.springer.com/article/10.1007/BF00592937)__.  First, we will read in the data and plot it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#Read in data
df = pd.read_csv('data/Sierra_nevada_granodiorite.txt',sep=',',header=0)

#Plot data
fig, ax = plt.subplots(1)
ax.scatter(df.rbsr8786, df.srsr8786)

#we will need to manually change the y range, else it's a bit too zoomed out
ax.set_ylim([0.706,0.708])

#we can label axes, here is the notation for putting mathematical expressions in labels
# if you know LaTeX you will be familiar with this, you just need an 'r' before the text
ax.set_xlabel(r'$^{87}Rb/^{86}Sr$')
ax.set_ylabel(r'$^{87}Sr/^{86}Sr$');

ax.grid(ls=':');

Now, we have a huge range of options for how to fit a line to the above data.  After all, we are just asking the question 
>what choice of model parameters minimise the distance between our model and the data?

which is a very general problem.  The human brain is quite good at evaluating how effective a model is in these simple cases, which is what you have relied upon previously when hand drawing lines through graphed data.  

The formal method, however, when faced with x-y data and straight line fitting is __linear least squares regression__.  A visualisation of what least squares regression is doing is included below.

![lsr](img/least-squares-regression-line.jpg)

Least squares regression is trying to minimise the summed area of those blue squares, which just represent the squared vertical (i.e., y-axis-parallel) distance between each data point and the model line.  Representing this mathematically, least squares regression is minimising the equation

\begin{align}
S &= \sum_{i=1}^n{\big[y_i - y'_i\big]^2}\\\\
y'_i &= f(x_i) = mx_i + c,
\end{align}

where, $y_i$ is the y-value of the $i$'th data point, and $f(x_i)$ is our model evaluated at $x_i$, the x-location of the $i$'th data point.

There are tools to minimise $S$ for us in Python, although if you want to see the maths behind solving the linear least squares regression problem start __[here](https://en.wikipedia.org/wiki/Least_squares)__.  

We will start by using `scipy`'s inbuilt function `scipy.stats.linregress`.  Linear regression problems are so common, that these inbuilt methods don't even require us to pass our own function to the solver, they are built to expect a $y=mx+c$ type problem.  As with everything `scipy`/`numpy`, there is __[good documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)__ explaining what we need to give this function and what it is going to give back to us.  The function simply wants
- $\mathbf{x}$
- $\mathbf{y}$

two arrays of equal length defining the x and y locations of each data point.  It will return an object with attributes giving us the best fit slope, intercept and other statistics (we access them with a dot, e.g. `res.slope`), check __[the documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)__ for details.

Let's use this function, plotting the output and putting some annotations onto the graph to tell us the age.

In [ ]:
import scipy.stats as sts
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#read in data
df = pd.read_csv('data/Sierra_nevada_granodiorite.txt',sep=',',header=0)

#let's call our linear regression function and store the result to an object called 'res'
res = sts.linregress(df.rbsr8786, df.srsr8786)

#now we are going to need our own function for a straight line so that we can plot
# the model result given an x value, m and c.
def lfunc(x, m, c):
    """Return y = m*x + c, the value of a straight line at x."""
    y = m*x + c
    return y

#let's plot this solution to see how well it has done
# first plotting the data...
#Plot data, changing the symbol style to look more sciency
fig, ax = plt.subplots(1)
ax.scatter(df.rbsr8786, df.srsr8786, facecolor='white', edgecolor='black')

#we will need to manually change the y range, else it's a bit too zoomed out
ax.set_ylim([0.706,0.708])

#plot on our model result
# first creating some x values, from the minimum x and maximum x values the plot has automatically assigned
xlim = ax.get_xlim()
x = np.linspace(xlim[0], xlim[1], 100)
#pass these x values to our function with the slope and intercept found from our fitting
y = lfunc(x, res.slope, res.intercept)

#now we are ready to plot our solution
ax.plot(x,y, linestyle='-', color='black')

#define lambda, decay constant, note we _cannot_ call this variable 'lambda' as Python reserves that word
lmbd = 1.393e-11
age = np.log(res.slope + 1)/lmbd
text = f'age: {age/1e6:.1f} Myr'
#ax.text() writes text on the plot.  transform=ax.transAxes means that the position, (0.9, 0.1), is a fraction
# of the width and height of the axes, measured from the bottom left corner, rather than in data units
ax.text(0.9,0.1, text, transform=ax.transAxes, horizontalalignment='right',verticalalignment='bottom')

#we can label axes, here is the notation for putting mathematical expressions in labels
# if you know LaTeX you will be familiar with this, you just need an 'r' before the text
ax.set_xlabel(r'$^{87}Rb/^{86}Sr$')
ax.set_ylabel(r'$^{87}Sr/^{86}Sr$');

ax.grid(ls=':');

**Your turn:** `res` holds the results of the fit.  Use it to get the slope and the intercept of the best fit line.

In [ ]:
slope = ___
intercept = ___
check('slope', slope)
check('intercept', intercept)

We were able to fit this model to our data without even providing an initial guess at the slope or intercept. For more complex model fitting problems however, where we have a complex function with more parameters, we would have had to use a different fitting algorithm that would have required an initial guess near the right solution to have got to the right answer.

The age obtained from the __[original paper](https://link.springer.com/article/10.1007/BF00592937)__ is copied below for comparison.

<img src="./img/kistler_excerpt.png" width="400">

Our result is a little different to what was obtained by __[Kistler et al. (1986)](https://link.springer.com/article/10.1007/BF00592937)__.  What did we not do in our fitting that we should have done?

Part of the difference comes from the decay constant. We used $\lambda = 1.393\times10^{-11}\,\mathrm{yr}^{-1}$, a more recent value, whereas studies from the 1980s used the older value of $1.42\times10^{-11}\,\mathrm{yr}^{-1}$.

**Your turn:** calculate the age (in Myr) that our fit gives if we use the older decay constant, $\lambda = 1.42\times10^{-11}\,\mathrm{yr}^{-1}$.  Does this explain all of the difference from Kistler et al.?

In [ ]:
lmbd_old = 1.42e-11
age_old_Myr = ___
check('old lambda age', age_old_Myr)

**Fix the bug:** this is meant to calculate the age from the slope, using the isochron equation, but the answer is nonsense.  Compare it with the equation at the top of the notebook, and fix it.

In [ ]:
lmbd = 1.393e-11
age_Myr = np.log(res.slope)/lmbd / 1e6
check('age', age_Myr)

## 5.2 Model fitting with errors
***

Now, a key thing we forgot in our solution above was that the data have errors on them.  It turns out that this makes the job of fitting a model quite a bit more tricky, especially when there are errors both on x and y, _and_ the errors are correlated between the x and y axes as is the case with isochron data (both axes have $^{86}\mathrm{Sr}$ as a common denominator).  Fitting such data fully is beyond the scope of this exercise.  But below we will plot the data with their errors, and then, in section 5.2.1, use the errors on the y values in the fit.

Note, that the errors on the $^{87}$Rb/$^{86}$Sr ratios are in percent (i.e., are relative to the Rb/Sr value itself), so need to be transformed to give absolute errors.

We will use `ax.errorbar` as our plotting tool, the arguments you need to pass to control the appearance are a bit different to usual here, so __[check the documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.errorbar.html)__.

**Your turn:** the cell below plots the data with their error bars, but it's missing the best fit line.  Add it, copying what we did in the first plot with a fitted line (in section 5.1.2).

In [ ]:
#-------------same as above---------------------------#
import scipy.stats as sts
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = pd.read_csv('data/Sierra_nevada_granodiorite.txt',sep=',',header=0)

res = sts.linregress(df.rbsr8786, df.srsr8786)

def lfunc(x, m, c):
    """Return y = m*x + c, the value of a straight line at x."""
    y = m*x + c
    return y

fig, ax = plt.subplots(1)
#-----------------------------------------------------#

#make plot with error bars
ax.errorbar(df.rbsr8786, df.srsr8786, yerr=df.srsr_1sig, xerr=(df.rbsr_1sigpc*df.rbsr8786/100),
            color='white', marker='o', markeredgecolor='black', linestyle='None',
           ecolor='black')

#we will need to manually change the y range, else it's a bit too zoomed out
# comment out the line below to see what matplotlib automatically sets as the y-range
ax.set_ylim([0.706,0.708])

#Your turn: plot the best fit line through the data, as we did in the first plot with a fitted line (section 5.1.2).
# (make some x values with np.linspace, calculate their y values with lfunc, and plot them)


#define lambda, the decay constant, note we _cannot_ call this variable 'lambda' as Python reserves that word
lmbd = 1.393e-11
age = np.log(res.slope + 1)/lmbd
text = f'age: {age/1e6:.1f} Myr'
#ax.text() writes text on the plot.  transform=ax.transAxes means that the position, (0.9, 0.1), is a fraction
# of the width and height of the axes, measured from the bottom left corner, rather than in data units
ax.text(0.9,0.1, text, transform=ax.transAxes, horizontalalignment='right',verticalalignment='bottom')

#we can label axes, here is the syntax for putting mathematical expressions in labels
# if you know LaTeX you will be familiar with this, you just need an 'r' before the text
ax.set_xlabel(r'${^{87}Rb/^{86}Sr}$')
ax.set_ylabel(r'${^{87}Sr/^{86}Sr}$');

ax.grid(ls=':');

### 5.2.1 Using the errors in the fit

We can't fully deal with errors on both axes here, but we *can* use the errors on the y values, with `scipy.optimize.curve_fit` (which we will meet properly in section 5.3).  Its `sigma=` argument gives the 1 sigma error of each y value, so points with smaller errors count for more in the fit: this is called a **weighted** fit.  `absolute_sigma=True` tells it these are real, measured uncertainties.

`curve_fit` gives back the best fit parameters (`popt`) and their **covariance matrix** (`pcov`).  The square roots of the numbers along the diagonal of `pcov` are the 1 sigma uncertainties on each parameter, which we can turn into an uncertainty on the age.

In [ ]:
import scipy.optimize as opt

#fit our straight line function, weighting each point by its y error
popt, pcov = opt.curve_fit(lfunc, df.rbsr8786, df.srsr8786, sigma=df.srsr_1sig, absolute_sigma=True)
#the 1 sigma uncertainties on the slope and intercept
perr = np.sqrt(np.diag(pcov))

#convert the slope, and the slope plus its error, into ages
age = np.log(popt[0] + 1)/lmbd
age_err = np.log(popt[0] + perr[0] + 1)/lmbd - age
print(f'age: {age/1e6:.1f} +/- {age_err/1e6:.1f} Myr')
print(f'initial 87Sr/86Sr: {popt[1]:.5f} +/- {perr[1]:.5f}')

#how well does the line fit, given the errors? The MSWD (mean square weighted deviation) is about 1
# if the points scatter about the line by about as much as their errors suggest
residuals = df.srsr8786 - lfunc(df.rbsr8786, popt[0], popt[1])
mswd = np.sum((residuals/df.srsr_1sig)**2) / (len(df) - 2)
print(f'MSWD: {mswd:.1f}')

Every point in this dataset has the same y error, so weighting doesn't change the line, and we get the same age as before.  What we gain is an uncertainty on the age.

But the MSWD is well above 1: the points scatter about the line more than their errors suggest.  So either the errors are underestimated, or the samples don't all share exactly the same age and initial composition.  Either way, the uncertainty above is probably too small.  A common fix is to multiply it by $\sqrt{\mathrm{MSWD}}$.

We have also ignored the x errors.  Once multiplied by the slope of the line, they are about ten times smaller than the y errors, so here they make little difference.  Fitting data with errors on both axes properly needs a different kind of regression (for example a 'York' regression).

**Your turn:** multiply the age uncertainty by $\sqrt{\mathrm{MSWD}}$, and give the answer in Myr.

In [ ]:
age_err_scaled_Myr = ___
check('scaled error', age_err_scaled_Myr)

## 5.3 Minimisation algorithms
***

The linear regression problem described above is a special case of minimisation, which as a reminder is where we are looking to minimise the distance between data and a model.  Let's now look at a slightly more complicated example, where we are no longer expecting a simple straight line to fit our data.  This example will introduce us to a more general case of minimisation.



### 5.3.1 Scientific background

We are going to develop an Fe-Ti oxide thermometer.  At equilibrium, the structure of ilmenite ($\mathsf{FeTiO_3}$) depends on the temperature, $T$, and on an ordering parameter, $Q$, which describes how the Fe and Ti cations are arranged: $Q = 0$ means they are completely mixed, and $Q = 1$ means they are fully separated onto alternate layers.

$T$ and $Q$ are related by
\begin{equation}
T = T_c−aQ^2−(T_c−a)Q^{n−2},
\end{equation}
where the constants $T_c$, $a$ and $n$ have to be found by fitting experimental data.  Each ilmenite composition has its own set of constants.

> **For the curious:** in the experiments, oxides with compositions between pure ilmenite and hematite are synthesised at known temperatures, and their ordering, $Q$, is then measured with neutron diffraction.

### 5.3.2 Data reconnaissance

Now, let's read in the experimental data for ilmenite and plot that up.  One trick with this data file (open `data/ilmenite_data_simple.txt` and have a look), is that there are multiple ilmenite compositions.  These each need to be fitted separately.  We will start by plotting each ilmenite dataset with a different coloured symbol so that we can see the different trends the data describe.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#Read in the data
# note, that this file is separated by tabs rather than commas (which is the default assumption when
# read_csv is used), so we specify this explicitly using 'sep='
df = pd.read_csv('data/ilmenite_data_simple.txt', sep='\t', header=0)

#We need to store the unique mineral names so we can plot and fit these datasets separately
minerals = df.composition.unique()

#start plotting by initialising a subplot and storing the figure and axis objects
fig, ax = plt.subplots()

#loop over the minerals, plotting each one separately
for m in minerals:
    #we use 'df.composition==m' to select specifically the data entries corresponding to the
    # specific mineral
    x = df.Q[df.composition==m]
    y = df.temp[df.composition==m]

    ax.scatter(x, y, label=m)

ax.set_xlabel('Q');
ax.set_ylabel('T (K)');
ax.legend();

Notice how matplotlib automatically adjusted the colour of symbols/lines when we repeatedly plot onto the same axes.  So all we needed to do was to use separate plotting commands to plot the different ilmenite data.

Plotting the data like this illuminates an important feature: some of the low temperature data look a little odd, not falling along a smooth transition from the high temperature data.  This might be because the low temperature experiments failed to equilibrate (because the timescales for equilibration were too long).  We need to take this into account when performing our fit to the data, and exclude the low temperature data.

> When fitting data to extract model parameters, always visualise the data and model fit to make sure things are working as expected.

### 5.3.3 Fitting a model

To make this a minimisation problem, we want to write a fitting function that will calculate a value for $T$ for a given input value of $Q$ and a particular choice of $T_c$, $a$ and $n$.  All these values will be passed to the function.  Let's start by writing a function for $T$.

In [ ]:
#Define fitting function
def ilmenite_T(Q, Tc, a, n):
    """Return the ilmenite temperature for ordering parameter Q, given the constants Tc, a and n."""
    return Tc - a*Q**2 - (Tc - a)*Q**(n-2)

Now we want to take the pieces of code we have from above and combine them to fit a model to the different ilmenite curves and from that identify the parameters that go into our thermometer equation.

For this we are going to use `scipy.optimize.curve_fit`, find the __[documentation here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)__ for the full details of how to use it.  This is a general and powerful tool for fitting a 'curve' (or, in other words a 'model') to data.  It takes the function we have defined above, `ilmenite_T`, the x and y coordinates of our data (i.e., Q and T values), and passes guesses for the constants we want to determine to the function.  It then calculates the misfit between the model and the data and adjusts its guess for the values of the constants to try and improve the misfit.  It returns the optimal values of the parameters and other information about the success of the fitting exercise (see the documentation for details).

A key difference from the user perspective between `curve_fit` and the `linregress` tool we used previously, is that `curve_fit` requires an initial guess of what the parameter values are.  We will use
\begin{align}
T_c &= 1200\\
a &= 1\\
n &= 6,
\end{align}
which work.  In practice, you may need to experiment at picking several different values before you have a set of initial guesses that allow the minimisation algorithm to find the best fit parameter values.  And to know whether a 'good fit' has been found, you will need to be visualising the results (data and model prediction), emphasising again that plotting the data and model is very important.

The code below loops through each of the ilmenite compositions and uses `scipy.optimize.curve_fit` to fit a model to each of them, plotting the model and the data on a graph.

In [ ]:
#---assume we have run the cells above that read in the data etc.

#import scipy.optimize
import scipy.optimize as opt
import numpy as np

#Define a temperature threshold below which the data suggest the experiments haven't
# reached equilibrium
T_thresh = 1000

#set up our plot
fig, ax = plt.subplots()

#initial guesses for Tc, a and n
Tc0 = 1200
a0 = 1
n0 = 6

#loop over the different ilmenite compositions
for m in minerals:
    #define a list of True False values, telling us which data we want to include
    # here, combining the constraint from temperature and mineral name
    inc = (df.composition==m) & (df.temp>T_thresh)

    #select _all_ the data for this composition
    x = df.Q[df.composition==m]
    y = df.temp[df.composition==m]

    #plot all the data as grey points (with no label, so they don't appear in the legend)
    ax.scatter(x, y, c='grey')
    #plot the data we are fitting as coloured symbols
    sc = ax.scatter(x[inc], y[inc], label=m)

    #now fit the data passing specifically the data we want to fit, i.e., using 'inc' to select
    popt, pcov = opt.curve_fit(ilmenite_T, x[inc], y[inc], p0=[Tc0, a0, n0])
    #the fitted parameters are the whole point of the thermometer, so let's print them with their uncertainties.
    # pcov is the 'covariance matrix' of the parameters, and the square roots of its diagonal are their
    # 1 sigma uncertainties (estimated here from how much the data scatter about the curve)
    perr = np.sqrt(np.diag(pcov))
    print(f'{m}: Tc = {popt[0]:.0f} +/- {perr[0]:.0f} K, a = {popt[1]:.0f} +/- {perr[1]:.0f}, n = {popt[2]:.1f} +/- {perr[2]:.1f}')

    #plot the model as a thin line over the full range of Q. We create evenly spaced Q values
    # with np.linspace, because the data's Q values aren't in order, which would make the line
    # double back on itself. We also give the line the same colour as the symbols.
    Q = np.linspace(x.min(), x.max(), 100)
    ax.plot(Q, ilmenite_T(Q, popt[0], popt[1], popt[2]), linewidth=1, color=sc.get_facecolor()[0])

ax.set_xlabel('Q')
ax.set_ylabel('T (K)')
ax.legend();

**Your turn:** what happens if `curve_fit` starts from a bad guess?  Try fitting the last composition again (`x[inc]` and `y[inc]` still hold its data), starting from `p0=[100, 1, 1]`.  If the fit fails, `curve_fit` gives an error: the `try`/`except` lets us catch that error and print a message, instead of stopping.  (If you run the cell before filling in the blank, it also prints 'The fit failed', with a message about missing arguments: that's because of the blank, not the bad guess.)

In [ ]:
try:
    popt_bad, pcov_bad = opt.curve_fit(ilmenite_T, x[inc], y[inc], p0=___)
    print('The fit worked:', popt_bad)
except Exception as e:
    print('The fit failed:', e)

# Independent coding
***



> **Aim:** To use a minimisation algorithm to identify the optimal pole of rotation between South America and Africa.

Plate tectonics makes a simple prediction you can test: transform faults should become parallel, horizontal lines on an oblique Mercator projection whose pole is the Euler pole of rotation between the two plates.  You will use the fracture zones of the South Atlantic to find the pole between South America and Africa that does this best.

> **For the curious:** McKenzie and Parker first did this test for the North Pacific in a classic 1967 paper (__[summary here](https://blogs.egu.eu/divisions/ts/2020/09/30/ts-must-read-papers-mckenzie-and-parker-1967/)__).

The equations to project longitude and latitude coordinates into $x$ and $y$ of the oblique Mercator projection are:

\begin{align}
x &= \arctan{\Bigg(\frac{(\tan(\phi)\cos(\phi_\mathrm{p}) + \sin(\phi_\mathrm{p})\sin(\lambda-\lambda_\mathrm{c}))}{\cos(\lambda-\lambda_\mathrm{c})}\Bigg)}\\\\
y &= \frac{1}{2}\ln\Bigg(\frac{1+A}{1-A}\Bigg)
\end{align}
where
\begin{align}
A &= \sin(\phi_\mathrm{p})\sin(\phi) - \cos(\phi_\mathrm{p})\cos(\phi)\sin(\lambda-\lambda_\mathrm{c})\\\\
\lambda_\mathrm{c} &= \lambda_\mathrm{p}+90.
\end{align}

In these equations $\phi$ is latitude and $\lambda$ is longitude in degrees (careful: here $\lambda$ means longitude, not the decay constant from section 5.1; both uses are standard in their own fields), subscript $p$ indicates the location of the pole of projection (pole of rotation) and symbols with no subscript represent the location of the point being projected.

To begin with, follow through the code below to plot the ridge segments and fracture zones in the region using a standard Mercator projection (i.e., the special case of the oblique Mercator where the pole is at 90N).  _Note: you will need to use elements of this code when writing your own solution._

In [ ]:
#the usual imports
import cartopy as ctp
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#let's get our figure set up, with subplot_kw passing the map projection (and its central longitude) to the axes
fig, ax = plt.subplots(figsize=(10,6), subplot_kw={'projection': ccrs.Mercator(central_longitude=-13)})
#these coordinates roughly bound our region of interest
lonmin, lonmax, latmin, latmax = -60, -5, -6, 20
ax.set_extent([lonmin, lonmax, latmin, latmax])

#let's add the oceans this time for fun
feature = ctp.feature.NaturalEarthFeature(
        name='ocean', category='physical',
        scale='110m',
        edgecolor='#000000', facecolor='lightblue', zorder=0)
ax.add_feature(feature)

#add the land (the 'land' layer is made of filled shapes, whereas 'coastline' is just lines)
feature = ctp.feature.NaturalEarthFeature(
        name='land', category='physical',
        scale='110m',
        edgecolor='#000000', facecolor='tan', zorder=2)
ax.add_feature(feature)

#add gridlines
gl = ax.gridlines(linestyle=':', color='black',
                 draw_labels=True, zorder=1)
gl.top_labels = False
gl.right_labels = False

#now for the ridges,
# each ridge in the file is represented by two pairs of lat-lon values
dr = pd.read_csv('data/gale_ridge_segs.csv',header=0, index_col=None)
#we should now restrict the ridges to those in the geographical region we are interested in
# (the condition is split over two lines; Python carries on reading until the brackets close)
dr = dr[(dr.Lat1<latmax) & (dr.Lat1>latmin) & (dr.Lat2<latmax) & (dr.Lat2>latmin) &
        (dr.Lon1<lonmax) & (dr.Lon1>lonmin) & (dr.Lon2<lonmax) & (dr.Lon2>lonmin)].reset_index(drop=True)

#now let's plot the ridge segments. We could loop through the rows of the dataframe, plotting one
# segment at a time, but ax.plot() can draw them all at once: if we give it 2D arrays, it draws each
# column as a separate line. np.vstack stacks the start and end points into two rows, so each column
# holds the start and end of one segment.
#remember, we need to use 'transform=ccrs.Geodetic()' to let the plot know that we are giving
# it lon-lat data, rather than plot (i.e., x-y) coordinates.
ax.plot(np.vstack([dr.Lon1, dr.Lon2]), np.vstack([dr.Lat1, dr.Lat2]), transform=ccrs.Geodetic(),
        color='orangered')

#now we can read in fracture zones and plot those
df = pd.read_csv('data/fracture_zones.csv',header=0, index_col=0)

#again, we plot all the fracture zones at once
ax.plot(np.vstack([df.lon1, df.lon2]), np.vstack([df.lat1, df.lat2]), transform=ccrs.Geodetic(),
        color='white');


Reassuringly, these fracture zones are somewhat misoriented with respect to the lines of latitude, but they are not wildly off: this is an important observation, just knowing this tells us that the 'answer' is probably a pole of rotation at reasonably high latitude.

Now, the task is to write a program to find the pole of rotation that fits these fracture zones.

Identifying that a pole at high latitude probably fits these fracture zones not only helps us evaluate whether we have a sensible solution once we have run our code, but also gives us a good idea of what to use for the initial guess, or for how to bound the parameter space our algorithm searches over.  All solvers will require some indication of what values the model parameters can possibly take.  In this case there are some important and obvious constraints: we know we are looking for a pole location lying on the surface of the Earth, so its latitude, $\phi_\mathrm{p}$ is restricted to $-90\le{}\phi_\mathrm{p}\le{}90$ and longitude, $\lambda_\mathrm{p}$, to $-180\le{}\lambda_\mathrm{p}\le{}180$.  What's more, we only need to find the 'north' end of the pole (there will of course be two solutions!), so we can restrict our search domain further.

Bounding the problem as tightly as possible with prior information is extremely important, as solvers can otherwise struggle to find the 'best' solution, amongst many good, but not quite as good, alternatives.

This brings us to a key step in posing our problem such that a computer can solve it: deciding how to express the 'fitness' of the solution, i.e., 'how good is the current pole at flattening out the transform faults?'  All our transform faults are expressed as pairs of longitude, latitude coordinates (the i'th transform fault is defined by [$\lambda^i_1$, $\phi^i_1$] and [$\lambda^i_2$, $\phi^i_2$]), and when we have the right solution the y values in the transformed coordinates for each transform fault should nearly be equal to each other.  This means we are looking to minimise

\begin{equation}
    S = \Bigg(\dfrac{\sum_{i=1}^n{\big[y^i_1-y^i_2\big]^2}}{n}\Bigg)^{0.5}.
\end{equation}

I.e., the root mean squared misfit of the data from the ideal scenario.

Choosing the right 'solver' for your problem is something of an art.  For now, focus on using a __[dual annealing solver](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.dual_annealing.html#scipy.optimize.dual_annealing)__ from `scipy`.  Read the description of what input the solver wants very carefully:
- It wants you to give it a function to which it can pass guesses about where the pole could be, and for that function to return to it a single number telling it how bad that guess is (the misfit), then it will update its guesses and try again, stopping when it thinks it can't do any better.
- It also wants 'bounds', that constrain the parameter space.  These are the latitude and longitude values that the pole could take, we have looked at this above.
- The final essential thing we will need to pass the solver is the additional arguments our minimisation function takes to run.  In our case this is going to be the data describing the locations of the fracture zones, so these can be projected onto an oblique Mercator projection using the current best-guess pole, and their deviation from horizontal parallel lines calculated.

You will see from the documentation for `scipy`'s 'dual_annealing' solver that there are many other options you can pass.  For now, accept that sensible defaults of these options have probably been chosen.

Work through these checkpoints:

1. **Set up.** Create a new notebook, called `Exercise5_solution`, in the course folder.  Read in the fracture zone data, __[fracture_zones.csv](data/fracture_zones.csv)__, into a dataframe called `df` (use `index_col=0`).
   Check: `len(df)` should be 50.
2. **The projection.** Write a function, for example `oblique_mercator(lat, lon, pole_lat, pole_lon)`, that uses the maths above to return the oblique Mercator `x` and `y` of a point (it should work for arrays of points too).  Test it with a single point at 10°N, 30°W and a pole at 90°N, 0°E: with the pole at the North Pole, $y$ is just the ordinary Mercator $y$.
   Check: the `y` your function gives should be about 0.17543.
3. **Test it on the data.** Project the fracture zones using a pole at 90°N, 0°E and plot them in a normal `matplotlib` x-y plot (don't use `Cartopy` for this!).  It should look like the map above.  If it doesn't, something has gone wrong.
4. **The misfit.** Write a function `misfit(pole, df)` that returns the root mean squared difference between the two projected $y$ values of each fracture zone (they are equal when the fracture zone is horizontal).  Test it with different poles, and check it with the published pole:
   Check: `misfit([62.5, -39.4], df)` should be about 0.0022047.
5. **Find the pole.** Call `scipy.optimize.dual_annealing(...)`, passing it your misfit function, the fracture zones and the bounds (Hints 2 and 3), plus `seed=42` so that it makes the same random choices as ours.  Store what it gives back in `res`.
   Check: `res.x` should be about [66.76, -36.10].
6. **Re-plot** the fracture zones projected about your best fit pole: they should now be horizontal lines!  Save the plot to a file, e.g. `fig.savefig('exercise5_best_fit.png', dpi=300)`.
7. **Compare** your answer with a __[published estimate](https://doi.org/10.1029/94GL02118)__ (their Table 2).

_Hint 1: Remember numpy trig functions take radians not degrees._

_Hint 2: You will need to pass the pandas dataframe containing the fracture zones to the solver as `.dual_annealing(..., args=(df,))`. (`args` must be a tuple; the comma in `(df,)` makes a tuple with one item in it.)_

_Hint 3: you will need to set the bounds for the dual annealing algorithm as `bounds = list(zip([0, -180],[90, 180]))`._

_Hint 4: when you calculate $x$, use `np.arctan2(numerator, denominator)` rather than `np.arctan(numerator/denominator)`. `arctan2` looks at the signs of the numerator and denominator separately, so it can tell which quadrant the angle is in; `arctan` only sees their ratio, and can't. If your projected points end up split between the left and right edges of the plot, $x$ has wrapped round from $\pi$ to $-\pi$: try a different longitude for your test pole._

### Going further
- Think about why our result is different to that of the published Africa-South America Pole.  How could what we did be improved to increase accuracy of plate rotation parameters?
- Adjust the other parameters setting __[how the dual annealing solver works](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.dual_annealing.html#scipy.optimize.dual_annealing)__.  In particular, `maxiter` and `initial_temp`.  Can you get the solver to _fail_ to find the best solution?
- Investigate whether __[other solvers](https://docs.scipy.org/doc/scipy/reference/optimize.html)__ work as well as the dual annealing solver we used.
- Try changing the function we are minimising, does this affect the result?